Placeholder notebook for implementation trials/tests

In [1]:
import torch

ashape = (3, 1, 4, 4)

a = torch.cat((torch.ones(ashape).unsqueeze(dim=0), 2 * torch.ones(ashape).unsqueeze(dim=0)), dim=0)

In [16]:
a.shape

torch.Size([2, 3, 1, 4, 4])

In [19]:
b = a.chunk(2, dim=0)

b[0].squeeze(dim=0), b[1].squeeze(dim=0)

(tensor([[[[1., 1., 1., 1.],
           [1., 1., 1., 1.],
           [1., 1., 1., 1.],
           [1., 1., 1., 1.]]],
 
 
         [[[1., 1., 1., 1.],
           [1., 1., 1., 1.],
           [1., 1., 1., 1.],
           [1., 1., 1., 1.]]],
 
 
         [[[1., 1., 1., 1.],
           [1., 1., 1., 1.],
           [1., 1., 1., 1.],
           [1., 1., 1., 1.]]]]),
 tensor([[[[2., 2., 2., 2.],
           [2., 2., 2., 2.],
           [2., 2., 2., 2.],
           [2., 2., 2., 2.]]],
 
 
         [[[2., 2., 2., 2.],
           [2., 2., 2., 2.],
           [2., 2., 2., 2.],
           [2., 2., 2., 2.]]],
 
 
         [[[2., 2., 2., 2.],
           [2., 2., 2., 2.],
           [2., 2., 2., 2.],
           [2., 2., 2., 2.]]]]))

In [1]:
a = 'cuda'
b = 'cuda:0'

def extract_devicetype(device: str) -> list[str]:
    return device.split(':')


tuple(map(extract_devicetype, (a, b)))

(['cuda'], ['cuda', '0'])

In [6]:
a = torch.zeros(10)

a.mean(), a.mean(dim=0)

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [8]:
list(range(10)), list(range(1, 10)), list(range(1, 11))

([0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [1, 2, 3, 4, 5, 6, 7, 8, 9],
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

In [14]:
list(enumerate(range(10))), list(enumerate(range(1, 11), start=6))

([(0, 0),
  (1, 1),
  (2, 2),
  (3, 3),
  (4, 4),
  (5, 5),
  (6, 6),
  (7, 7),
  (8, 8),
  (9, 9)],
 [(6, 1),
  (7, 2),
  (8, 3),
  (9, 4),
  (10, 5),
  (11, 6),
  (12, 7),
  (13, 8),
  (14, 9),
  (15, 10)])

In [16]:
import torch
import torch.nn as nn

b, c = 1, 1

dshape = (b, c, 11, 15)
mshape = (b, c, 21, 25)

pad_n, pad_m = dshape[-2] - 1, dshape[-1] - 1
pad_mode = torch.nn.ConstantPad2d((pad_m, pad_m, pad_n, pad_n), 0)

detector = torch.zeros(dshape)
mask = pad_mode(torch.zeros(mshape))

sky = torch.conv2d(mask, detector)

sshape = sky.shape
assert mask.shape == torch.Size([1, 1, 41, 53]), f'Current padded mask shape is: {mask.shape}'
assert sshape == torch.Size([1, 1, 31, 39]), f'Current sky shape is: {sshape}'

In [ ]:
dspath: str = '/mnt/d/PhD_AASS/Coding/Images_fits/irosdiffsn_dataset_ID1_nels100.pickle'

data: dict[str, Any] = load_pickle(dspath)

In [12]:
data['camera'].keys()

dict_keys(['specs', 'upsampling', 'arrays', 'data_shape', 'bins'])

In [ ]:
field = data['data']
sgs, gtpars = map(torch.tensor, (tuple(field.values())))

In [11]:
sgs.shape

torch.Size([100, 308, 1264])

In [5]:
import torch

img = torch.poisson(0.1 * torch.ones((3, 1, 5, 5)))

mask = (img > 0).float()
nels = mask.sum(dim=(-2, -1), keepdim=True)

mu = img.sum(dim=(-2, -1), keepdim=True) / nels
std = torch.square(mask * (img - mu))
std = std.sum(dim=(-2, -1), keepdim=True) / nels
std = std.sqrt().clamp(min=1e-6)

# b = mask * (img - mu) / std
b = torch.where(mask.bool(), (img - mu) / std, torch.zeros_like(img))


b.mean(dim=(-2, -1)), b.std(dim=(-2, -1))

(tensor([[0.],
         [0.],
         [0.]]),
 tensor([[0.],
         [0.],
         [0.]]))

In [2]:
b = (a - a.median()) / a.std()

b.sum(), b.mean(), b.median(), b.std(), b.max(), b.min()

(tensor(66.7125),
 tensor(0.0002),
 tensor(0.),
 tensor(1.),
 tensor(6.0073),
 tensor(-3.1618))

In [3]:
a = torch.poisson(20 * torch.ones((500, 300, 1200)))
b = a.amax(dim=(-2, -1))

b.mean(), b.std()

(tensor(44.0540), tensor(1.6026))

In [1]:
from pkdev.dataset import get_dataset, get_dataloaders

# dirpath: str = '/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Diffusion/IROSdiffusion_datasets'
dirpath: str = '/mnt/d/PhD_AASS/Coding/Images_fits'

data = get_dataset(dirpath)

Loading Data: 0it [00:00, ?it/s]

Loading...


Loading Data: 1it [00:00,  2.09it/s]

Data loaded!
Loading...


Loading Data: 2it [00:00,  2.02it/s]

Data loaded!
Loading...


Loading Data: 3it [00:01,  1.91it/s]

Data loaded!


In [6]:
import torch
from pkdev.dataset import normalise_sgs

b, n, m = 10, 300, 1200
fps = torch.ones((b, n, m))

for i in range(b):
    # Select random bounding box coordinates
    y1, y2 = sorted(torch.randint(0, n, (2,)).tolist())
    x1, x2 = sorted(torch.randint(0, m, (2,)).tolist())
    fps[i, y1:y2, x1:x2] = 0.0

sgs = torch.poisson(20 * torch.ones_like(fps)) * fps

norm = normalise_sgs(sgs, fps)

idx = 0

for idx in range(b):
    mask = (fps[idx] > 0)
    print(f"Mean: {norm[idx][mask].mean():.4f}, Std: {norm[idx][mask].std():.4f}")

Mean: -0.0000, Std: 1.0000
Mean: -0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000
Mean: -0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000
Mean: 0.0000, Std: 1.0000


In [2]:
sgs = data.shadowgrams

sgs[5].squeeze(dim=0).sum()

tensor(0.0002)

In [14]:
from pkdev.dataset import normalise_sgs


norm_sgs = normalise_sgs(sgs)

res = sgs - norm_sgs

res.amax(dim=(-2, -1)), res.amin(dim=(-2, -1))

(tensor([4.2573, 0.7817, 0.3965, 0.1120, 0.0191, 0.4420, 0.5416, 1.3669, 0.6395,
         0.5343]),
 tensor([-0.3960, -1.2012, -1.6832, -1.8275, -2.0281, -1.6198, -1.3702, -1.2693,
         -1.2363, -1.2162]))

In [20]:
import numpy as np

rng = np.random.default_rng()

a = rng.poisson(2, (10, 300, 1200))

b = (a - a.mean(axis=(-2, -1), keepdims=True)) / a.std(axis=(-2, -1), keepdims=True)

a.mean(axis=(-2, -1)), a.std(axis=(-2, -1)), b.mean(axis=(-2, -1)), b.std(axis=(-2, -1))

(array([2.00236944, 2.00632778, 1.99891111, 2.000475  , 1.99828056,
        1.99990556, 2.00053056, 2.00101389, 2.004575  , 1.99992778]),
 array([1.41481347, 1.41411494, 1.41441544, 1.41384319, 1.41241906,
        1.41363401, 1.41619688, 1.41544911, 1.41508487, 1.41247419]),
 array([-1.34845221e-16, -1.07726174e-16,  4.01851392e-17,  1.82214737e-16,
        -3.61587303e-17, -3.62376795e-17, -1.29358253e-16, -1.02949747e-16,
         2.68427256e-17,  3.00006933e-17]),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]))

In [23]:
_, n, m = a.shape

c = a[:, :, :int(0.42 * m)]

d = (c - c.mean(axis=(-2, -1), keepdims=True)) / c.std(axis=(-2, -1), keepdims=True)

c.mean(axis=(-2, -1)), c.std(axis=(-2, -1)), d.mean(axis=(-2, -1)), d.std(axis=(-2, -1))

(array([2.00539683, 2.00719577, 1.99944444, 1.9976455 , 1.99746032,
        1.99851852, 1.99708995, 2.00202381, 2.00641534, 2.00338624]),
 array([1.41786037, 1.41302092, 1.41485868, 1.40896873, 1.41524444,
        1.41432502, 1.41286305, 1.41613291, 1.41670349, 1.41146633]),
 array([ 1.06393436e-16, -1.61657871e-17,  4.33280689e-17,  4.06024420e-17,
         5.73321520e-17, -8.03120063e-17,  5.90239204e-17, -7.57536303e-17,
         7.78213472e-17, -6.84226338e-17]),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]))